# Momentum
## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Construct a momentum signal** — the standard (12, 1) past return
2. **Build a long-short momentum portfolio** following the recipe from L13
3. **Diagnose momentum crashes** — the most important risk in the strategy
4. **Apply volatility scaling to momentum** to mitigate crash risk
5. **Audit AI-generated momentum code** — the 1-month skip, holding period, rebalancing

## 📋 TOC
1. [Setup](#setup)  2. [The Momentum Signal](#signal)
3. [Pitfall Checklist](#pitfalls)  4. [Live Demo: Building Momentum](#demo)
5. [Momentum Crashes](#crashes)  6. [Risk-Managed Momentum](#riskmgmt)
7. [🎯 Challenge: Crash Magnitudes](#challenge)
8. [Submission](#submit)  9. [Key Takeaways](#takeaways)

---
## 🛠️ Setup <a id="setup"></a>

In [ ]:
#@title Setup
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize']=[10,5]; plt.rcParams['font.size']=11
import warnings; warnings.filterwarnings('ignore')
print("✅ Loaded")

---
## The Momentum Signal <a id="signal"></a>

The **standard momentum signal**, dating to Jegadeesh & Titman (1993):

$$\text{MOM}_{i,t} = \text{cumulative return of stock } i \text{ from month } t-12 \text{ to month } t-2$$

Two key features:
- **12-month lookback** — captures medium-term trend
- **Skip the most recent month** — that one tends to *reverse* (short-term reversal)

Then form quintile portfolios on MOM each month and go long top minus bottom.

---
## 🛡️ Pitfall Checklist <a id="pitfalls"></a>

| | Pitfall | What goes wrong | 🔍 How to detect |
|---|---------|-----------------|-------------------|
| 1 | **Forgetting the 1-month skip** | Including month $t-1$ flips the sign because of short-term reversal | Standard MOM = `cum_ret[t-12:t-2]`, not `[t-12:t-1]` |
| 2 | **Not lagging the signal vs return** | Computing portfolios on $t$ data, holding them at $t$ | Form portfolios at end of $t$, earn return in $t+1$ |
| 3 | **Holding period mismatch** | Signal computed monthly but portfolio held longer | Most academic momentum is monthly rebalanced |
| 4 | **Ignoring transaction costs** | Momentum has 150-200% turnover/year | Real-world momentum has slim margins after costs |
| 5 | **Backtest bias** | Looking only at recent decades misses 1932, 2009 crashes | Always include 1929 + 2008 if your data goes back that far |

---
## 🔄 Live Demo: Building Momentum <a id="demo"></a>

> **🤖 AI prompt:**
>
> *"Given a panel of monthly stock returns (long format: columns permno, date, ret),
> compute the (12, 1) momentum signal: for each (permno, date), the cumulative
> log-return from date-12 to date-2 months ago. Use shift(2) and rolling(11).sum()
> on log returns. Lag the signal by one month before sorting."*

In [ ]:
# Simulated mini-panel demo (3 fake stocks over 60 months)
np.random.seed(42)
T, N = 60, 3
dates = pd.date_range('2020-01-31', periods=T, freq='ME')
data = []
for permno in [10001, 10002, 10003]:
    rets = np.random.normal(0.01, 0.06, T)
    for d, r in zip(dates, rets):
        data.append({'permno': permno, 'date': d, 'ret': r})
panel = pd.DataFrame(data).sort_values(['permno', 'date'])

# Log returns
panel['logret'] = np.log1p(panel['ret'])

# 12-1 momentum: cumulative log-return from t-12 to t-2 (so 11 months, skipping t-1)
def mom_signal(g):
    g = g.sort_values('date').copy()
    # shift(2) makes the window end at t-2; rolling(11).sum() captures 11 months
    g['mom'] = g['logret'].shift(2).rolling(11).sum()
    return g

panel = panel.groupby('permno', group_keys=False).apply(mom_signal)
print(panel.tail(10))

---
## Momentum Crashes <a id="crashes"></a>

Momentum has the most extreme drawdowns of any "anomaly":
- **November 2008**: -25% in a single month
- **March-April 2009**: ~-50% over 6 weeks
- **April 2020**: another sharp negative month

**Why?** Momentum buys winners and shorts losers. After a market crash,
the "losers" (deep cyclicals) rebound massively as investors pile back in.
Momentum is short these — and gets crushed.

Crashes are predictable in one sense: they happen after long market
drawdowns. Volatility is high. Correlations are high.

> **💡 The fundamental tradeoff**
>
> Momentum has a great long-run Sharpe (~0.5-1.0). But the drawdowns are
> brutal enough that many investors can't stomach it. **Real-world deploy
> requires risk management.**

---
## Risk-Managed Momentum <a id="riskmgmt"></a>

The simplest fix: **scale momentum exposure inversely to realized vol**:

$$w^{MOM}_t = c \cdot \frac{1}{\hat{\sigma}^{MOM}_{t-1}^2}$$

This is the same vol-timing logic from Lecture 8 — but applied to the
momentum factor instead of the market.

Barroso & Santa-Clara (2015) showed this single tweak cuts the maximum
drawdown roughly in half while keeping most of the Sharpe.

---
## 🎯 Challenge: Crash Magnitudes <a id="challenge"></a>

The historical momentum factor (UMD from Ken French's data) has the following stylized facts in our sample:

- Mean annual return: ~7%
- Annual vol: ~15%
- Sharpe: ~0.47
- Worst month: ~-30%
- Skewness: -1.5 (heavily left-tailed)

### Q1 — Vol-scaled Sharpe improvement

Barroso-Santa-Clara found that vol-scaling raises momentum's Sharpe from
~0.47 to ~0.62 while cutting the worst month from -30% to ~-15%.

> **📌 Required:**
> ```python
> raw_sharpe          = 0.47
> raw_worst_month     = -0.30
> vol_scaled_sharpe   = ____   # Barroso-Santa-Clara number
> vol_scaled_worst    = ____
> ```

In [ ]:
raw_sharpe       = 0.47
raw_worst_month  = -0.30

vol_scaled_sharpe = ____
vol_scaled_worst  = ____
print(f"Sharpe improvement: {raw_sharpe:.2f} → {vol_scaled_sharpe:.2f}")
print(f"Worst month:        {raw_worst_month:.2%} → {vol_scaled_worst:.2%}")

### Q2 — Premium per unit of crash risk

Crash risk = absolute value of the worst month. Compute the ratio of
annual mean to crash risk for both versions.

> **📌 Required:**
> ```python
> raw_return       = 0.07
> raw_crash_ratio  = ____   # raw_return / |raw_worst_month|
> scaled_crash_ratio = ____  # 0.07 * (vol_scaled_sharpe / raw_sharpe) / |vol_scaled_worst|
> ```

In [ ]:
raw_return = 0.07

raw_crash_ratio    = ____
scaled_crash_ratio = ____
print(f"Raw return per unit crash:    {raw_crash_ratio:.2f}")
print(f"Scaled return per unit crash: {scaled_crash_ratio:.2f}")

### Q3 — Memo

Max 5 sentences. Recommend whether your fund should run momentum, and if so,
whether raw or vol-scaled. Cite (i) Sharpe, (ii) crash risk, (iii) what kind
of investor each version is appropriate for.

In [ ]:
MEMO = """Write your memo here."""
print(MEMO)

---
## 📤 Submission <a id="submit"></a>

In [ ]:
# === 📤 SUBMISSION CELL ===
import json, base64, hashlib, datetime as dt
required = ["vol_scaled_sharpe", "vol_scaled_worst", "raw_crash_ratio", "scaled_crash_ratio", "MEMO"]
missing = [v for v in required if v not in dir()]
if missing: raise NameError(f"\n❌ Missing: {missing}")
payload = {"assignment": "Momentum_AI",
    "ts": dt.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip()}
blob = json.dumps(payload, sort_keys=True)
token = f"UG54::{hashlib.sha256(blob.encode()).hexdigest()[:8]}::{base64.b64encode(blob.encode()).decode()}"
print("="*72); print(token); print("="*72)

---
## 🧠 Key Takeaways <a id="takeaways"></a>
1. **Standard MOM = (12, 1) past return.** Skip the most recent month.
2. **Momentum's worst flaw is its crash risk.** -25% to -50% drawdowns in market reversals.
3. **Vol-scaling roughly halves crash risk** while keeping most of the Sharpe.
4. **Always lag the signal.** The recipe is: form portfolios at end of $t$, earn return in $t+1$.
5. **AI handles the rolling-window code. You catch the skip-month bug and the lookahead.**